In [ ]:
pip install catboost

In [ ]:
import mlflow
import mlflow.catboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==============================================================================
# 0. CONFIGURACIÓN DEL EXPERIMENTO ÚNICO
# ==============================================================================
experiment_name = "/modelos-hipervalorizacion-bogota"
mlflow.set_experiment(experiment_name)

print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
df = pd.read_csv(matriz)

# Separar datos supervisados vs inferencia
df_supervised = df[df['v_ref_2026'].notnull()].copy()
df_inference = df[df['v_ref_2026'].isnull()].copy()

# Preparar Matriz X e y 
y_sup = df_supervised['v_ref_2026']
cols_a_excluir = ['barrio', 'geometry', 'v_ref_2015', 'v_ref_2026', 'num_listings_airbnb', 'area_km2']
X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])

# CatBoost maneja variables categóricas de forma nativa
cat_features = ['localidad']
for col in cat_features:
    if col in X_sup.columns:
        X_sup[col] = X_sup[col].astype(str)

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)

# Hiperparámetros iniciales para CatBoost
params = {
    "iterations": 600,
    "learning_rate": 0.04,
    "depth": 6,
    "loss_function": "RMSE",
    "random_seed": 42
}

# ==============================================================================
# RUN 1: CATBOOST SIN LOGARITMO (Precios Brutos)
# ==============================================================================
with mlflow.start_run(run_name=f"CatBoost_Sin_Log"):
    mlflow.log_params(params)
    mlflow.log_param("target_transform", "Ninguna")

    # Entrenamiento directo sobre los pesos colombianos
    cat_bruto = CatBoostRegressor(**params, verbose=0) # verbose=0 para no llenar la consola
    cat_bruto.fit(X_train, y_train, cat_features=cat_features)

    # Evaluación Train
    y_pred_train_bruto = cat_bruto.predict(X_train)
    mae_train_bruto = mean_absolute_error(y_train, y_pred_train_bruto)
    r2_train_bruto = r2_score(y_train, y_pred_train_bruto)

    # Evaluación Test
    y_pred_test_bruto = cat_bruto.predict(X_test)
    mae_test_bruto = mean_absolute_error(y_test, y_pred_test_bruto)
    rmse_test_bruto = np.sqrt(mean_squared_error(y_test, y_pred_test_bruto))
    r2_test_bruto = r2_score(y_test, y_pred_test_bruto)

    # Registro de métricas con los nombres exactos de RF
    mlflow.log_metric("mae_train_pesos", mae_train_bruto)
    mlflow.log_metric("mae_test_pesos", mae_test_bruto)
    mlflow.log_metric("r2_train", r2_train_bruto)
    mlflow.log_metric("r2_test", r2_test_bruto)
    mlflow.log_metric("rmse_test_pesos", rmse_test_bruto)
    
    mlflow.catboost.log_model(cat_bruto, "catboost_model_bruto")


# ==============================================================================
# RUN 2: CATBOOST CON LOGARITMO 
# ==============================================================================
with mlflow.start_run(run_name=f"CatBoost_Con_Log"):
    mlflow.log_params(params)
    mlflow.log_param("target_transform", "Logaritmo Natural")

    # Transformación de la variable objetivo
    y_train_log = np.log(y_train)

    # Entrenamiento
    cat_log = CatBoostRegressor(**params, verbose=0)
    cat_log.fit(X_train, y_train_log, cat_features=cat_features)

    # Evaluación Train (Revertida)
    y_pred_train_log = cat_log.predict(X_train)
    y_pred_train_rev = np.exp(y_pred_train_log)
    mae_train_rev = mean_absolute_error(y_train, y_pred_train_rev)
    r2_train_rev = r2_score(y_train, y_pred_train_rev)

    # Evaluación Test (Revertida)
    y_pred_test_log = cat_log.predict(X_test)
    y_pred_test_rev = np.exp(y_pred_test_log)
    mae_test_rev = mean_absolute_error(y_test, y_pred_test_rev)
    rmse_test_rev = np.sqrt(mean_squared_error(y_test, y_pred_test_rev))
    r2_test_rev = r2_score(y_test, y_pred_test_rev)

    # Registro de métricas con los nombres exactos de RF
    mlflow.log_metric("mae_train_pesos", mae_train_rev)
    mlflow.log_metric("mae_test_pesos", mae_test_rev)
    mlflow.log_metric("r2_train", r2_train_rev)
    mlflow.log_metric("r2_test", r2_test_rev)
    mlflow.log_metric("rmse_test_pesos", rmse_test_rev)

    mlflow.catboost.log_model(cat_log, "catboost_model_log")